# SOC-Triage-Gym v2 — End-to-End Demo

An OpenEnv-style multi-agent environment for **Security Operations Center (SOC) alert triage**.

This notebook runs **on CPU in under two minutes** with no GPU, no LLM API key, and no Colab. It uses the in-process `SOCEnvironment` and the deterministic `HeuristicBaselineAgent` to demonstrate:

1. **Solo episode** — manual action calls (`enrich_indicator` → `query_logs` → `classify_alert`).
2. **Oracle baseline sweep** — heuristic agent on `phishing`, `lateral_movement`, `queue_management` × 3 seeds.
3. **Team mode** — multi-agent run on `team_phishing_escalation` (Tier-1 → Tier-2 → Manager phases).
4. **HTTP API check** — the same env exposed via FastAPI / `TestClient`.
5. **Plot** — bar chart of per-task mean scores.

For full GRPO training (Unsloth + TRL on GPU), see `soc_triage_gym_v2_training.ipynb`.

In [ ]:
import os, sys, warnings, json

warnings.filterwarnings("ignore")
sys.path.insert(0, os.getcwd())

from models import SOCAction, ActionType, IndicatorType, AlertClassification, LogSource
from server.environment import SOCEnvironment
from baseline_agent import HeuristicBaselineAgent
from scenarios import SCENARIO_REGISTRY
from graders import GRADER_REGISTRY

print("Tasks available:")
for tid in SCENARIO_REGISTRY:
    print(f"  - {tid}")
print(f"\nGraders registered: {len(GRADER_REGISTRY)}")

## 1. Solo episode walkthrough

Reset the `phishing` task with `seed=42`, then take three explicit actions to show the API surface.

In [ ]:
env = SOCEnvironment()
obs = env.reset(task_id="phishing", seed=42)

alert = obs.alert_queue[0]
print(f"Episode {obs.episode_id[:8]}…  task={obs.task_id}  budget={obs.investigation_budget}")
print(f"Alert: {alert.alert_id} | severity={alert.severity.value} | title={alert.title!r}")
print(f"Indicators: {dict(alert.indicators)}\n")

ip = alert.indicators["ip"][0]
obs = env.step(SOCAction(
    action_type=ActionType.ENRICH_INDICATOR,
    indicator=ip,
    indicator_type=IndicatorType.IP,
    query_alert_id=alert.alert_id,
))
er = obs.enrichment_results[-1]
print(f"[step {obs.step}] enrich({ip}) -> malicious={er.malicious} score={er.threat_score:.2f} reward={obs.reward:+.3f}")

obs = env.step(SOCAction(
    action_type=ActionType.QUERY_LOGS,
    log_source=LogSource.EMAIL_GATEWAY,
    query_alert_id=alert.alert_id,
))
print(f"[step {obs.step}] query_logs(email_gateway) -> {len(obs.log_results)} entries  reward={obs.reward:+.3f}")

obs = env.step(SOCAction(
    action_type=ActionType.CLASSIFY_ALERT,
    alert_id=alert.alert_id,
    classification=AlertClassification.TRUE_POSITIVE,
    confidence=0.9,
))
print(f"[step {obs.step}] classify -> {obs.investigations[alert.alert_id].classification.value}  reward={obs.reward:+.3f}")
print(f"\nCumulative reward so far: {obs.cumulative_reward:+.3f}")

## 2. Oracle baseline sweep

Run the deterministic `HeuristicBaselineAgent` to completion on three solo tasks, three seeds each. The environment computes the final score via the registered grader and exposes it as `obs.task_score` when `done=True`.

In [ ]:
def run_solo_episode(task_id: str, seed: int, max_steps: int = 200) -> float:
    env = SOCEnvironment()
    agent = HeuristicBaselineAgent()
    obs = env.reset(task_id=task_id, seed=seed)
    steps = 0
    while not obs.done and steps < max_steps:
        action_dict = agent.next_action(obs.model_dump())
        obs = env.step(SOCAction(**action_dict))
        steps += 1
    return float(obs.task_score or 0.0), steps

TASKS = ["phishing", "lateral_movement", "queue_management"]
SEEDS = [42, 123, 256]

results = {t: [] for t in TASKS}
for task_id in TASKS:
    for seed in SEEDS:
        score, steps = run_solo_episode(task_id, seed)
        results[task_id].append(score)
        print(f"  {task_id:20s} seed={seed:<4d}  steps={steps:<3d}  score={score:.3f}")

print("\nMean score per task:")
mean_scores = {t: sum(s) / len(s) for t, s in results.items()}
for t, m in mean_scores.items():
    print(f"  {t:20s}  {m:.3f}")

## 3. Team mode (multi-agent)

`team_phishing_escalation` runs a three-phase episode: **Tier-1 triage** → **Tier-2 response** → **Manager oversight**. Each role has its own action set; transitions happen on `phase_complete` or when the phase budget runs out. We use the role-aware oracle from `train_grpo.py` since the heuristic baseline is solo-only.

In [ ]:
from train_grpo import oracle_action

env = SOCEnvironment()
obs = env.reset(task_id="team_phishing_escalation", seed=42, mode="team")
print(f"mode={obs.episode_mode.value}  starting role={obs.current_role.value}  phase={obs.current_phase.value}")
print(f"phase budget remaining: {obs.phase_steps_remaining}\n")

phases_seen = []
step_count = 0
while not obs.done and step_count < 200:
    phase = obs.current_phase.value if obs.current_phase else None
    if phase and (not phases_seen or phases_seen[-1] != phase):
        phases_seen.append(phase)
        print(f"-- entering phase: {phase} (role={obs.current_role.value}) --")
    action_dict = oracle_action(obs.model_dump())
    obs = env.step(SOCAction(**action_dict))
    step_count += 1

print(f"\nDone in {step_count} steps. Phases visited: {phases_seen}")
print(f"Final task_score: {obs.task_score:.3f}")
if obs.team_reward_breakdown:
    b = obs.team_reward_breakdown
    print(f"Team reward breakdown: tier1={b.tier1_individual:+.2f}  tier2={b.tier2_individual:+.2f}  "
          f"manager={b.manager_individual:+.2f}  shared={b.team_shared:+.2f}  total={b.total:+.2f}")

## 4. HTTP API check (FastAPI `TestClient`)

The same environment is exposed over HTTP via [`server/app.py`](server/app.py). `TestClient` lets us hit the endpoints in-process — same wire format as a real client, no uvicorn subprocess needed.

In [ ]:
from fastapi.testclient import TestClient
from server.app import app

with TestClient(app) as client:
    print("GET  /health  ->", client.get("/health").json())

    reset_resp = client.post("/reset", json={"task_id": "phishing", "seed": 42}).json()
    print(f"POST /reset   -> task={reset_resp['task_id']}  alerts={len(reset_resp['alert_queue'])}  "
          f"budget={reset_resp['investigation_budget']}")

    aid = reset_resp["alert_queue"][0]["alert_id"]
    ip = reset_resp["alert_queue"][0]["indicators"]["ip"][0]
    step_resp = client.post("/step", json={
        "action_type": "enrich_indicator",
        "indicator": ip,
        "indicator_type": "ip",
        "query_alert_id": aid,
    }).json()
    print(f"POST /step    -> step={step_resp['step']}  reward={step_resp['reward']:+.3f}  "
          f"enrichments={len(step_resp['enrichment_results'])}")

    tasks = client.get("/tasks").json()
    print(f"GET  /tasks   -> {len(tasks)} tasks registered")

## 5. Plot baseline scores

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
names = list(mean_scores.keys())
values = [mean_scores[n] for n in names]
bars = ax.bar(names, values, color=["#4C9AFF", "#FF8B00", "#36B37E"])
ax.set_ylim(0, 1.0)
ax.set_ylabel("Mean task score (3 seeds)")
ax.set_title("SOC-Triage-Gym v2 — Heuristic Baseline")
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.02, f"{v:.2f}", ha="center", fontsize=10)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("soc_triage_gym_v2_results.png", dpi=120)
plt.show()
print("Saved soc_triage_gym_v2_results.png")

## Next steps

- **GRPO training** on top of this baseline — see [`soc_triage_gym_v2_training.ipynb`](soc_triage_gym_v2_training.ipynb) (requires GPU).
- **Full benchmark sweep** — `python benchmark.py` runs all tasks × multiple seeds and emits a markdown table with a determinism check.
- **LLM-driven inference** — `python inference.py` plays episodes with a real Llama / GPT-class model via the OpenAI-compatible API.
- **Run the server standalone** — `uvicorn server.app:app --host 0.0.0.0 --port 7860`, then point any HTTP client at `/reset`, `/step`, `/state`.